# ArmorStart → ArmorPowerFlex 755 Parameter Conversion
Parses ArmorStart DeviceNet HTML export(s), applies conversion rules, and writes a formatted Excel workbook — one tab per device.

**Usage:**
1. Set `HTML_PATH` in Cell 2 to your RSNetWorx/DeviceNet HTML export file.
2. Set `OUTPUT_PATH` to the desired Excel output location.
3. Run All Cells (`Kernel → Restart & Run All`).

---
## Conversion Rules Reference
*(Source: last cell of original APF_Dnet.ipynb)*

```
Rule Setup
-Motor Control
     - Motor Configuration
         - Motor Type = "AC Rotary Induction"
         - Motor Control Mode = P225
             0=V/Hz
             1=SVC
         - Maximum PWM Frequency: P191

     - Frequency Control
         - Curve Type: P184
             0 = Custom V/Hz
             1-4 = Variable Torque
             9-14 = Constant Torque
         - Start Boost: P185 when P184 = 0.
         - Break Voltage: P186 when P184 = 0.
         - Brake Frequency: P187 when P184 = 0

     - Motor Nameplate:
         - Rated Voltage: P131
         - Rated Frequency: P132
         - Rated Speed: User Input
         - Rated Power: User Input
         - Full Load Amps: P226

     - Motor Protection
         - Thermal Overload Power Up Behavior: "Assume Motor Cold"
         - Overload Current: P133
         - Derate Frequency: "0.00"
         - Current Limit 1: P189
         - Current Limit 2: P218

     - Load Loss
         - Load Loss Monitor : "Disabled"

     - Shear Pin Function
         - Shear Pin 1: "Disabled"
         - Shear Pin 2: "Disabled"

     - Flying Start: P196
          0 = Disabled
          1 = Enabled

      - Auto Restart
          - Enable Auto Restart: "Enabled" if P192 not equal to 0
          - Auto Restart Retries: P192
          - Auto Restart Delay: P193

  - Encoder Feedback
      - Electrical Interface Type: "Disabled Encoder Feedback"

  - Velocity Control
      - Minimum Velocity: Calculated. (P134 / P132) * Rated Speed / 60
      - Maximum Velocity: Calculated  (P135 / P132) * Rated Speed / 60
        - Ramp Presets
          - 1
              - Acceleration Time: P139
              - Deceleration Time: P140
              - S-Curve %: P183
          - 2
              - Acceleration Time: P167
              - Deceleration Time: P168
              - S-Curve %: P183
        - Output Frequency
          - Minimum Frequency: P134
          - Maximum Frequency: P135

  - Stop Control
      - Stop Mode: P137
          1,5 = "Coast"
          0,4 = "Ramp"
          2,3,6,7 = DC Brake
      - Dynamic Brake
          - Resistor Select: P182
              0 = "Not Specified"
              1 = "External Normal Duty"
              2-99 = "External No Protection"
          - DC Bus Voltage Threshold: "770"
      - EM Brake
          - Control Mode: "Automatic or Manual Control" when P137 is not equal to 1 or 5
          - Off Delay: P260
          - On Delay: P261

 - IO Configuration
     - Input Configuration (Inputs 0-3 all share same source):
         - Off -> On: P30
         - On -> Off: P31
```

In [34]:
# ── Imports ───────────────────────────────────────────────────────────────────
from bs4 import BeautifulSoup
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import re
import os

In [35]:
# ── Configuration ─────────────────────────────────────────────────────────────
# TODO: Update paths before running
HTML_PATH   = r"C:\Users\USLanceSt\OneDrive - NESTLE\Documents\01_Scripts\ArmorPowerFlexDnet\00_Inputs\Line1_Zone1_2.html"
OUTPUT_PATH = r"C:\Users\USLanceSt\OneDrive - NESTLE\Documents\01_Scripts\ArmorPowerFlexDnet\01_Outputs\APF_Converted.xlsx"

In [36]:
# ── Constants ─────────────────────────────────────────────────────────────────

# ArmorStart parameters that are read-only (monitor only, not configurable on APF)
READ_ONLY = {1, 2, 3, 4, 5, 6, 7, 17, 18, 21, 27, 29, 56, 57, 58, 59, 60, 61, 62, 74, 75, 76, 77}

# Stop Mode (P137) value → APF string
STOP_MODE_MAP = {
    0: "0 - Ramp",
    1: "1 - Coast",
    2: "2 - DC Brake",
    3: "3 - DC Brake",
    4: "4 - Ramp",
    5: "5 - Coast",
    6: "6 - DC Brake",
    7: "7 - DC Brake",
}

# Motor Control Mode (P225) value → APF string
CTRL_MODE_MAP = {
    "0": "0 - V/Hz",
    "1": "1 - SVC",
}

MANUAL = "⚠ Manual Input Required"

In [37]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def extract_params(table_tag):
    """Extract (Instance, Parameter Name, Current Value) rows from an ArmorStart param table."""
    rows = []
    for tr in table_tag.find_all("tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) == 3:  # skip header rows that use <th>
            rows.append(cells)
    return rows


def strip_units(s):
    """Remove trailing unit strings such as ' Hz', ' %', ' A', ' V'."""
    return re.sub(r'\s+[A-Za-z%]+$', '', str(s).strip())


def get_val(df, instance):
    """Return the stripped Current Value for a given Instance number, or None."""
    row = df[df["Instance"] == instance]
    if row.empty:
        return None
    return strip_units(row.iloc[0]["Current Value"])


def safe_int(v):
    """Convert a value to int, returning None on failure."""
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None


def resistor_label(v):
    """Map P182 (DB Resistor Select) integer to APF display string."""
    if v == 0:  return "0 - Not Specified"
    if v == 1:  return "1 - External Normal Duty"
    return f"{v} - External No Protection"


def curve_label(v):
    """Map P184 (Boost Select / Curve Type) integer to APF display string."""
    if v == 0:            return "0 - Custom V/Hz"
    if 1 <= v <= 4:       return f"{v} - Variable Torque"
    if 9 <= v <= 14:      return f"{v} - Constant Torque"
    return str(v)

In [38]:
# ── Parse HTML Export ─────────────────────────────────────────────────────────

with open(HTML_PATH, "r", encoding="iso-8859-1") as f:
    soup = BeautifulSoup(f, "html.parser")

# Locate every DEVICE_TITLE div that contains "ArmorStart"
device_titles = [
    d for d in soup.find_all(id="DEVICE_TITLE")
    if "ArmorStart" in d.get_text()
]
print(f"Found {len(device_titles)} ArmorStart device(s)")

all_dfs = []

for title_tag in device_titles:
    device_name = title_tag.get_text(strip=True)
    p_tag = title_tag.parent.find("p")
    table = p_tag.find("table") if p_tag else None

    if table is None:
        print(f"  ⚠  No parameter table found for: {device_name}")
        continue

    rows = extract_params(table)
    df = pd.DataFrame(rows, columns=["Instance", "Parameter Name", "Current Value"])
    df = df.apply(lambda col: col.str.strip())
    df["Instance"] = pd.to_numeric(df["Instance"], errors="coerce").astype("Int64")

    # Extract tag name from device string, e.g. "Address 10,  ArmorStart CNVR_70000" → "CNVR_70000"
    tag_match = re.search(r'ArmorStart\s+(\S+)', device_name)
    tag = tag_match.group(1) if tag_match else device_name

    # Extract DeviceNet address
    addr_match = re.search(r'Address\s+(\d+)', device_name)
    addr = addr_match.group(1) if addr_match else "?"

    df.insert(0, "Device", device_name)
    df.insert(1, "Tag",    tag)
    df.insert(2, "Addr",   addr)
    all_dfs.append(df)
    print(f"  ✓  {device_name}  →  {len(df)} params")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal rows: {combined.shape[0]}  |  Columns: {list(combined.columns)}")
display(combined)

Found 9 ArmorStart device(s)
  ✓  Address 10,  ArmorStart CNVR_70000  →  216 params
  ✓  Address 12,  ArmorStart CNVR_70001  →  216 params
  ✓  Address 14,  ArmorStart CNVR_70002  →  216 params
  ✓  Address 16,  ArmorStart CNVR_70003  →  216 params
  ✓  Address 18,  ArmorStart CNVR_70004  →  216 params
  ✓  Address 20,  ArmorStart CNVR_70005  →  216 params
  ✓  Address 22,  ArmorStart CNVR_70006  →  216 params
  ✓  Address 24,  ArmorStart CNVR_70007  →  216 params
  ✓  Address 26,  ArmorStart CNVR_70008  →  216 params

Total rows: 1944  |  Columns: ['Device', 'Tag', 'Addr', 'Instance', 'Parameter Name', 'Current Value']


,Device,Tag,Addr,Instance,Parameter Name,Current Value
0,"Address 10, ArmorStart CNVR_70000",CNVR_70000,10,1,Hdw Inputs,XXXXXXXX XXXX0101
1,"Address 10, ArmorStart CNVR_70000",CNVR_70000,10,2,Network Inputs,00000000 00000000
2,"Address 10, ArmorStart CNVR_70000",CNVR_70000,10,3,Network Outputs,X0000000 00000000
3,"Address 10, ArmorStart CNVR_70000",CNVR_70000,10,4,Trip Status,00000000 00000000
4,"Address 10, ArmorStart CNVR_70000",CNVR_70000,10,5,Starter Status,00100000 11110100
...,...,...,...,...,...,...
1939,"Address 26, ArmorStart CNVR_70008",CNVR_70008,26,212,Anlg In4-20mA Lo,0.0 %
1940,"Address 26, ArmorStart CNVR_70008",CNVR_70008,26,213,Anlg In4-20mA Hi,100.0 %
1941,"Address 26, ArmorStart CNVR_70008",CNVR_70008,26,214,Slip Hertz @ FLA,2.0 Hz
1942,"Address 26, ArmorStart CNVR_70008",CNVR_70008,26,215,Process Time Lo,0.00


In [39]:
# ── Excel Style Definitions ───────────────────────────────────────────────────

HDR_FILL  = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")  # column headers
GRP_FILL  = PatternFill("solid", start_color="2E75B6", end_color="2E75B6")  # group headers
SUB_FILL  = PatternFill("solid", start_color="BDD7EE", end_color="BDD7EE")  # sub-group headers
RO_FILL   = PatternFill("solid", start_color="F2F2F2", end_color="F2F2F2")  # read-only rows
MAN_FILL  = PatternFill("solid", start_color="FFF2CC", end_color="FFF2CC")  # manual input rows
TITLE_FILL= PatternFill("solid", start_color="DEEAF1", end_color="DEEAF1")  # title row

WHITE_BOLD = Font(name="Arial", bold=True, color="FFFFFF", size=10)
DARK_BOLD  = Font(name="Arial", bold=True, color="1F4E79", size=10)
NORMAL     = Font(name="Arial", size=10)
ITALIC     = Font(name="Arial", size=10, italic=True, color="595959")
TITLE_FONT = Font(name="Arial", bold=True, size=12, color="1F4E79")

thin = Side(style="thin", color="B8CCE4")
BDR  = Border(left=thin, right=thin, top=thin, bottom=thin)

CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",   vertical="center", wrap_text=True)

COLS  = ["APF Parameter Group", "APF Parameter Name", "APF Parameter #",
         "Source (ArmorStart Param)", "Value"]
COL_W = [28, 32, 18, 34, 36]

In [40]:
# ── Excel Sheet Builder ───────────────────────────────────────────────────────

def build_sheet(wb, tag, addr, dev_df):
    """Write one converted APF parameter sheet for a single device."""

    ws = wb.create_sheet(title=tag)
    ws.freeze_panes = "A3"

    # Convenience: get stripped value for an instance number
    def gv(inst):
        return get_val(dev_df, inst)

    # Row pointer (list so inner functions can mutate it)
    R = [3]

    # ── Cell writers ──────────────────────────────────────────────────────────
    def dcell(row, col, val, fill=None, font=NORMAL, align=LEFT):
        c = ws.cell(row=row, column=col, value=val)
        if fill: c.fill = fill
        c.font = font; c.border = BDR; c.alignment = align

    def group_hdr(label):
        ws.merge_cells(start_row=R[0], start_column=1, end_row=R[0], end_column=5)
        c = ws.cell(row=R[0], column=1, value=label)
        c.fill = GRP_FILL; c.font = WHITE_BOLD; c.border = BDR; c.alignment = LEFT
        R[0] += 1

    def sub_hdr(label):
        ws.merge_cells(start_row=R[0], start_column=1, end_row=R[0], end_column=5)
        c = ws.cell(row=R[0], column=1, value=label)
        c.fill = SUB_FILL; c.font = DARK_BOLD; c.border = BDR; c.alignment = LEFT
        R[0] += 1

    def apf(group, name, param_num, source, value, read_only=False, manual=False):
        fill  = RO_FILL if read_only else (MAN_FILL if manual else None)
        vfont = ITALIC if manual else NORMAL
        dcell(R[0], 1, group,     fill=fill)
        dcell(R[0], 2, name,      fill=fill)
        dcell(R[0], 3, param_num, fill=fill, align=CENTER)
        dcell(R[0], 4, source,    fill=fill)
        dcell(R[0], 5, value,     fill=fill, font=vfont)
        R[0] += 1

    # ── Title row ─────────────────────────────────────────────────────────────
    ws.merge_cells("A1:E1")
    t = ws["A1"]
    t.value = f"ArmorPowerFlex 755 — Parameter Conversion  |  {tag}  (ArmorStart Addr {addr})"
    t.font = TITLE_FONT; t.fill = TITLE_FILL; t.alignment = CENTER
    ws.row_dimensions[1].height = 22

    # ── Column header row ─────────────────────────────────────────────────────
    for ci, (col, w) in enumerate(zip(COLS, COL_W), 1):
        c = ws.cell(row=2, column=ci, value=col)
        c.fill = HDR_FILL; c.font = WHITE_BOLD; c.border = BDR; c.alignment = CENTER
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[2].height = 30

    # ══════════════════════════════════════════════════════════════════════════
    # MOTOR CONTROL
    # ══════════════════════════════════════════════════════════════════════════
    group_hdr("Motor Control")

    # ── Motor Configuration ───────────────────────────────────────────────────
    sub_hdr("  Motor Configuration")
    apf("Motor Control", "Motor Type", "—", "Fixed", "AC Rotary Induction")

    ctrl_raw = gv(225)  # P225 does not exist on ArmorStart -- returns None
    # Default to SVC when parameter is absent; map known values otherwise
    ctrl_val = CTRL_MODE_MAP.get(ctrl_raw, '1 - SVC') if ctrl_raw is not None else '1 - SVC'
    ctrl_src = 'ArmorStart P225 (Control Method)' if ctrl_raw is not None else 'P255 was Default'
    apf("Motor Control", "Motor Control Mode", "P225", ctrl_src, ctrl_val)

    apf("Motor Control", "Maximum PWM Frequency", "P191",
        "ArmorStart P191 (PWM Frequency)", gv(191))

    # ── Frequency Control ─────────────────────────────────────────────────────
    sub_hdr("  Frequency Control")
    curve_raw = gv(184)
    curve_int = safe_int(curve_raw) or 0
    apf("Motor Control", "Curve Type", "P184",
        "ArmorStart P184 (Boost Select)", curve_label(curve_int))

    # Start Boost / Break Voltage / Break Frequency only apply when P184 = 0 (Custom V/Hz)
    if curve_int == 0:
        apf("Motor Control", "Start Boost",     "P185",
            "ArmorStart P185 (Start Boost) [active: P184=0]",    gv(185))
        apf("Motor Control", "Break Voltage",   "P186",
            "ArmorStart P186 (Break Voltage) [active: P184=0]",  gv(186))
        apf("Motor Control", "Break Frequency", "P187",
            "ArmorStart P187 (Break Frequency) [active: P184=0]",gv(187))
    else:
        apf("Motor Control", "Start Boost",     "P185", "N/A — P184 ≠ 0", "Not Applicable")
        apf("Motor Control", "Break Voltage",   "P186", "N/A — P184 ≠ 0", "Not Applicable")
        apf("Motor Control", "Break Frequency", "P187", "N/A — P184 ≠ 0", "Not Applicable")

    # ── Motor Nameplate ───────────────────────────────────────────────────────
    sub_hdr("  Motor Nameplate")
    apf("Motor Control", "Rated Voltage",   "P131", "ArmorStart P131 (Motor NP Volts)", gv(131))
    apf("Motor Control", "Rated Frequency", "P132", "ArmorStart P132 (Motor NP Hz)",    gv(132))
    apf("Motor Control", "Rated Speed",     "—",    "User Input", MANUAL, manual=True)
    apf("Motor Control", "Rated Power",     "—",    "User Input", MANUAL, manual=True)
    apf("Motor Control", "Full Load Amps",  "P226", "ArmorStart P226 (Motor NP FLA)",   gv(226))

    # ── Motor Protection ──────────────────────────────────────────────────────
    sub_hdr("  Motor Protection")
    apf("Motor Control", "Thermal OL Power Up Behavior", "—",    "Fixed",                               "Assume Motor Cold")
    apf("Motor Control", "Overload Current",             "P133", "ArmorStart P133 (Motor OL Current)",  gv(133))
    apf("Motor Control", "Derate Frequency",             "—",    "Fixed",                               "0.00")
    apf("Motor Control", "Current Limit 1",              "P189", "ArmorStart P189 (Current Limit 1)",   gv(189))
    apf("Motor Control", "Current Limit 2",              "P218", "ArmorStart P218 (Current Limit 2)",   gv(218))

    # ── Load Loss ─────────────────────────────────────────────────────────────
    sub_hdr("  Load Loss")
    apf("Motor Control", "Load Loss Monitor", "—", "Fixed", "Disabled")

    # ── Shear Pin ─────────────────────────────────────────────────────────────
    sub_hdr("  Shear Pin Function")
    apf("Motor Control", "Shear Pin 1", "—", "Fixed", "Disabled")
    apf("Motor Control", "Shear Pin 2", "—", "Fixed", "Disabled")

    # ── Flying Start ──────────────────────────────────────────────────────────
    sub_hdr("  Flying Start")
    fs_int = safe_int(gv(196)) or 0
    apf("Motor Control", "Flying Start", "P196",
        "ArmorStart P196 (Flying Start)",
        "0 - Disabled" if fs_int == 0 else "1 - Enabled")

    # ── Auto Restart ──────────────────────────────────────────────────────────
    sub_hdr("  Auto Restart")
    ar_int = safe_int(gv(192)) or 0
    apf("Motor Control", "Enable Auto Restart",   "P192",
        "Enabled if P192 ≠ 0",
        "Enabled" if ar_int != 0 else "Disabled")
    apf("Motor Control", "Auto Restart Retries",  "P192", "ArmorStart P192 (Auto Rstrt Tries)", gv(192))
    apf("Motor Control", "Auto Restart Delay",    "P193", "ArmorStart P193 (Auto Rstrt Delay)", gv(193))

    # ══════════════════════════════════════════════════════════════════════════
    # ENCODER FEEDBACK
    # ══════════════════════════════════════════════════════════════════════════
    group_hdr("Encoder Feedback")
    apf("Encoder Feedback", "Electrical Interface Type", "—", "Fixed", "Disabled Encoder Feedback")

    # ══════════════════════════════════════════════════════════════════════════
    # VELOCITY CONTROL
    # ══════════════════════════════════════════════════════════════════════════
    group_hdr("Velocity Control")

    # Min/Max Velocity: formula string — replace [RatedSpeed] after manual entry
    p134 = gv(134); p135 = gv(135); p132 = gv(132)
    apf("Velocity Control", "Minimum Velocity", "—",
        "Calc: (Minimum Frequency / Rated Frequency) * Rated Speed / 60", MANUAL, manual=True)
    apf("Velocity Control", "Maximum Velocity", "—",
        "Calc: (Maximum Frequency / Rated Frequency) * Rated Speed / 60", MANUAL, manual=True)

    # ── Ramp Presets ──────────────────────────────────────────────────────────
    sub_hdr("  Ramp Presets")

    sub_hdr("    Ramp Preset 1")
    apf("Velocity Control", "Acceleration Time", "P139", "ArmorStart P139 (Accel Time 1)", gv(139))
    apf("Velocity Control", "Deceleration Time", "P140", "ArmorStart P140 (Decel Time 1)", gv(140))
    apf("Velocity Control", "S-Curve %",         "P183", "ArmorStart P183 (S Curve %)",    gv(183))

    sub_hdr("    Ramp Preset 2")
    apf("Velocity Control", "Acceleration Time", "P167", "ArmorStart P167 (Accel Time 2)", gv(167))
    apf("Velocity Control", "Deceleration Time", "P168", "ArmorStart P168 (Decel Time 2)", gv(168))
    apf("Velocity Control", "S-Curve %",         "P183",
        "ArmorStart P183 (S Curve % — same source as Preset 1)", gv(183))

    # ── Output Frequency ──────────────────────────────────────────────────────
    sub_hdr("  Output Frequency")
    apf("Velocity Control", "Minimum Frequency", "P134", "ArmorStart P134 (Minimum Freq)", gv(134))
    apf("Velocity Control", "Maximum Frequency", "P135", "ArmorStart P135 (Maximum Freq)", gv(135))

    # ══════════════════════════════════════════════════════════════════════════
    # STOP CONTROL
    # ══════════════════════════════════════════════════════════════════════════
    group_hdr("Stop Control")

    stop_int = safe_int(gv(137)) or 0
    apf("Stop Control", "Stop Mode", "P137",
        "ArmorStart P137 (Stop Mode)",
        STOP_MODE_MAP.get(stop_int, str(stop_int)))

    # ── Dynamic Brake ─────────────────────────────────────────────────────────
    sub_hdr("  Dynamic Brake")
    res_int = safe_int(gv(182)) or 0
    apf("Stop Control", "Resistor Select",         "P182", "ArmorStart P182 (DB Resistor Sel)", resistor_label(res_int))
    apf("Stop Control", "DC Bus Voltage Threshold", "—",    "Fixed",                             "770")

    # ── EM Brake ──────────────────────────────────────────────────────────────
    sub_hdr("  EM Brake")
    # Control Mode is set to Automatic/Manual when Stop Mode is NOT Coast (1 or 5)
    if stop_int not in {1, 5}:
        em_ctrl = "Automatic or Manual Control"
        em_src  = f"ArmorStart P137={stop_int} (not Coast)"
    else:
        em_ctrl = "Not Applicable (Coast Stop)"
        em_src  = f"ArmorStart P137={stop_int} (Coast)"
    apf("Stop Control", "EM Brake Control Mode", "—",    em_src,                          em_ctrl)
    apf("Stop Control", "EM Brake Off Delay",    "P260", "ArmorStart P260 (DB Time Off)", gv(260))
    apf("Stop Control", "EM Brake On Delay",     "P261", "ArmorStart P261 (DB Time On)",  gv(261))

    # ══════════════════════════════════════════════════════════════════════════
    # IO CONFIGURATION
    # ══════════════════════════════════════════════════════════════════════════
    group_hdr("IO Configuration")
    sub_hdr("  Input Configuration  (Inputs 0–3 share the same ArmorStart source)")

    for inp in range(4):
        sub_hdr(f"    Input {inp}")
        apf("IO Configuration", f"Input {inp} — Off → On Delay", "P30",
            "ArmorStart P30 (shared across all inputs)", gv(30))
        apf("IO Configuration", f"Input {inp} — On → Off Delay", "P31",
            "ArmorStart P31 (shared across all inputs)", gv(31))

    # ══════════════════════════════════════════════════════════════════════════
    # READ-ONLY REFERENCE
    # ══════════════════════════════════════════════════════════════════════════
    group_hdr("Read-Only Reference Parameters  (Monitor Only — Do Not Configure on APF)")

    ro_df = dev_df[dev_df["Instance"].isin(READ_ONLY)].sort_values("Instance")
    for _, ro_row in ro_df.iterrows():
        apf("Read-Only",
            ro_row["Parameter Name"],
            f"P{ro_row['Instance']}",
            f"ArmorStart P{ro_row['Instance']}",
            strip_units(ro_row["Current Value"]),
            read_only=True)

    # ── Legend ────────────────────────────────────────────────────────────────
    R[0] += 1
    legend_items = [
        (MAN_FILL, ITALIC,     "Yellow — Manual Input Required (Rated Speed, Rated Power, Min/Max Velocity)"),
        (RO_FILL,  NORMAL,     "Gray   — Read-Only / Reference  (do not configure in APF)"),
        (GRP_FILL, WHITE_BOLD, "Dark Blue — Parameter Group Header"),
        (SUB_FILL, DARK_BOLD,  "Light Blue — Sub-Group Header"),
    ]
    ws.merge_cells(start_row=R[0], start_column=1, end_row=R[0], end_column=5)
    lbl = ws.cell(row=R[0], column=1, value="LEGEND")
    lbl.font = Font(name="Arial", bold=True, size=10)
    R[0] += 1
    for fill, font, label in legend_items:
        ws.merge_cells(start_row=R[0], start_column=1, end_row=R[0], end_column=5)
        c = ws.cell(row=R[0], column=1, value=label)
        c.fill = fill; c.font = font; c.border = BDR
        R[0] += 1

    return ws

In [41]:
# ── Build Workbook ────────────────────────────────────────────────────────────

wb = Workbook()
wb.remove(wb.active)  # remove default blank sheet

# Get unique devices in order of appearance
device_list = combined[["Tag", "Addr"]].drop_duplicates()

for _, row in device_list.iterrows():
    tag  = row["Tag"]
    addr = row["Addr"]
    dev_df = combined[combined["Tag"] == tag].copy()
    build_sheet(wb, tag, addr, dev_df)
    print(f"  ✓  Built sheet: {tag}")

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

wb.save(OUTPUT_PATH)
print(f"\nSaved → {OUTPUT_PATH}")

  ✓  Built sheet: CNVR_70000
  ✓  Built sheet: CNVR_70001
  ✓  Built sheet: CNVR_70002
  ✓  Built sheet: CNVR_70003
  ✓  Built sheet: CNVR_70004
  ✓  Built sheet: CNVR_70005
  ✓  Built sheet: CNVR_70006
  ✓  Built sheet: CNVR_70007
  ✓  Built sheet: CNVR_70008

Saved → C:\Users\USLanceSt\OneDrive - NESTLE\Documents\01_Scripts\ArmorPowerFlexDnet\01_Outputs\APF_Converted.xlsx
